# 7장. RAG 시스템 평가 — 04. 커스텀 평가자 (Custom Evaluators)

## 학습 목표

- 커스텀 평가자 함수의 시그니처와 반환 형식 이해
- **규칙 기반**, **LLM 기반**, **임베딩 기반** 세 가지 커스텀 평가자 직접 구현
- 여러 평가자를 조합해 다차원 평가 수행

## 사전 지식

- 03번 노트북에서 LangSmith 기본 평가 경험
- 커스텀 평가자 함수 시그니처 이해

## 커스텀 평가자 기본 구조

모든 커스텀 평가자는 다음 형식을 따라야 해요:

```python
from langsmith.schemas import Run, Example

def my_evaluator(run: Run, example: Example) -> dict:
    # run.outputs  : 시스템이 생성한 출력 (평가 대상)
    # example.inputs  : 데이터셋의 입력
    # example.outputs : 데이터셋의 참조 답변 (선택적)
    
    score = ...  # 0~1 사이 값 계산
    return {"key": "metric_name", "score": score}
```

- `key`: LangSmith 대시보드에 표시될 지표 이름
- `score`: 0~1 사이 숫자 (또는 불리언 0/1)

> 🔑 **핵심 개념**: 커스텀 평가자의 시그니처는 딱 두 가지입니다 — `(Run, Example) -> dict`. `Run`에는 시스템이 생성한 출력이, `Example`에는 데이터셋의 입력과 참조 답변이 담깁니다. 이 구조만 이해하면 어떤 평가 로직이든 구현할 수 있습니다.

---

## 환경 설정

In [1]:
# 필요한 패키지 설치
# !pip install -qU langsmith langchain langchain-openai

In [ ]:
from dotenv import load_dotenv
import os

# 환경변수 로드
load_dotenv()

# macOS에서 FAISS 사용 시 OpenMP 중복 로드 방지
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# LangSmith 프로젝트 설정
os.environ["LANGSMITH_PROJECT"] = "04-Eval-Custom-Evaluators"

# LangSmith 설정 확인
print(f"LANGCHAIN_API_KEY: {'설정됨' if os.getenv('LANGCHAIN_API_KEY') else '미설정'}")
print(f"LANGSMITH_PROJECT: {os.getenv('LANGSMITH_PROJECT')}")

# ---------------------------------------------------
# [LangSmith UI 확인 방법]
# 1. https://smith.langchain.com 접속
# 2. 좌측 Projects 클릭
# 3. "04-Eval-Custom-Evaluators" 프로젝트 선택
# 4. 주안점: Experiments 탭 → 커스텀 평가자 점수 컬럼 확인
# ---------------------------------------------------

---

## 평가할 RAG 시스템 구축

In [2]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ---------------------------------------------------
# 트랜스포머 관련 한국어 컨텍스트 문서
# ---------------------------------------------------
documents = [
    Document(page_content=(
        "트랜스포머(Transformer)는 2017년 구글이 'Attention Is All You Need' 논문에서 "
        "제안한 신경망 아키텍처입니다. 기존의 순환 신경망(RNN)이나 합성곱 신경망(CNN) 없이 "
        "어텐션 메커니즘만으로 시퀀스를 처리합니다. 인코더와 디코더 각각 6개의 동일한 레이어를 "
        "쌓아 올린 구조이며, 모든 레이어의 출력 차원은 d_model = 512입니다."
    )),
    Document(page_content=(
        "셀프 어텐션(Self-Attention)은 시퀀스 내 모든 위치 간의 관계를 계산하는 메커니즘입니다. "
        "입력에서 쿼리(Query), 키(Key), 값(Value) 벡터를 생성하고, 쿼리와 키의 유사도로 "
        "어텐션 가중치를 계산합니다. 이를 통해 모델이 입력의 어떤 부분에 집중해야 하는지 학습합니다."
    )),
    Document(page_content=(
        "멀티헤드 어텐션(Multi-Head Attention)은 셀프 어텐션을 여러 개의 '헤드'로 병렬 수행하는 "
        "기법입니다. 각 헤드는 서로 다른 표현 부분공간(representation subspace)에서 정보를 추출합니다. "
        "논문에서는 8개의 헤드를 사용했으며, 각 헤드의 출력을 연결한 뒤 선형 변환을 적용합니다."
    )),
    Document(page_content=(
        "포지셔널 인코딩(Positional Encoding)은 트랜스포머에 단어의 위치 정보를 제공합니다. "
        "트랜스포머는 순환 구조가 없어 단어의 순서를 알 수 없으므로, 사인(sin)과 코사인(cos) "
        "함수 기반의 위치 벡터를 입력 임베딩에 더해줍니다."
    )),
    Document(page_content=(
        "트랜스포머의 인코더는 입력 시퀀스를 연속적인 표현으로 변환하고, 디코더는 이를 바탕으로 "
        "출력을 한 토큰씩 생성합니다. 디코더에는 마스크드 셀프 어텐션이 추가되어 미래 토큰 참조를 "
        "방지합니다. 인코더-디코더 어텐션으로 디코더가 입력의 관련 부분에 집중합니다."
    )),
    Document(page_content=(
        "트랜스포머는 RNN과 달리 시퀀스를 병렬로 처리할 수 있어 훈련 속도가 크게 빠릅니다. "
        "RNN은 순차 처리가 필수지만 트랜스포머는 모든 위치를 동시에 처리합니다. "
        "장거리 의존성도 더 효과적으로 포착할 수 있습니다."
    )),
    Document(page_content=(
        "스케일드 닷 프로덕트 어텐션은 쿼리와 키의 내적을 키 차원의 제곱근으로 나누어 스케일링합니다. "
        "스케일링 없이는 내적 값이 커져 소프트맥스의 기울기가 매우 작아집니다. "
        "스케일링 후 소프트맥스를 적용하여 가중치를 구하고, 값 벡터에 곱하여 출력을 얻습니다."
    )),
]

# 벡터 DB 생성
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever()

# 3단계: 프롬프트 정의
prompt = PromptTemplate.from_template(
    """주어진 컨텍스트를 바탕으로 질문에 답변하세요. 간결하고 구체적으로 답변하세요.

컨텍스트: {context}

질문: {question}

답변:"""
)

# 4단계: LLM 및 체인 생성
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG 시스템 구축 완료")

RAG 시스템 구축 완료


> 🎯 **강의 포인트**: 평가용 함수가 `answer`만이 아니라 `context`도 함께 반환하는 이유가 있습니다. 근거성(Groundedness) 평가자는 답변이 컨텍스트에 근거하는지 비교해야 하기 때문입니다. 평가자가 필요한 정보를 모두 `run.outputs`에서 꺼낼 수 있도록 설계하세요.

### 평가용 함수 정의

LangSmith `evaluate()` 함수는 `inputs -> outputs` 형식을 요구해요. RAG 체인의 답변뿐 아니라 **검색된 컨텍스트**도 함께 반환해서 평가자가 활용할 수 있도록 합니다.

> 💡 **실무 팁**: 규칙 기반 평가자는 개발 초기 빠른 반복에 최적입니다. LLM 호출이 없어서 비용이 0원이고, 결과가 항상 동일해서 디버깅이 쉽습니다. 먼저 규칙 기반으로 명확히 실패하는 케이스를 걸러낸 뒤, 경계 케이스에만 LLM 평가자를 사용하세요.

In [3]:
# ---------------------------------------------------
# 평가용 함수 정의 — 질문, 답변, 컨텍스트 모두 반환
# ---------------------------------------------------
# ============================================================
# TODO: 질문, 답변, 컨텍스트를 모두 반환하는 평가용 함수를 완성하세요
# 힌트: retriever.invoke(question)으로 docs를 검색하고, prompt | llm | parser 체인으로 답변 생성
# 예상 결과: {"question": ..., "answer": ..., "context": ...} 형식
# ============================================================

def rag_answer_with_context(inputs: dict) -> dict:
    """
    RAG 시스템에 질문하고 답변과 컨텍스트 반환
    
    Args:
        inputs: {"question": str} 형식의 딕셔너리
        
    Returns:
        {"question": str, "answer": str, "context": str} 형식의 딕셔너리
    """
    question = inputs["question"]
    
    # 1단계: 컨텍스트 검색 (한 번만 호출)
    docs = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in docs])
    
    # 2단계: 검색된 컨텍스트를 직접 전달하여 답변 생성
    # rag_chain을 재사용하면 내부에서 retriever를 다시 호출하므로,
    # prompt → llm → parser 체인을 직접 구성하여 동일 컨텍스트를 사용합니다.
    answer_chain = prompt | llm | StrOutputParser()
    answer = answer_chain.invoke({"context": context, "question": question})
    
    # 3단계: 질문, 답변, 컨텍스트 모두 반환
    return {
        "question": question,
        "answer": answer,
        "context": context
    }

---

## 1. 규칙 기반 커스텀 평가자

**LLM 호출 없이** 간단한 규칙으로 답변을 평가해요. 빠르고 비용이 들지 않아서 개발 초기 반복 평가에 유용합니다.

이 평가자는 세 가지 기준으로 점수를 매겨요:
- 답변 **길이** (너무 짧거나 길면 감점)
- **키워드** 포함 여부 (Transformer 관련 핵심 용어)
- **불필요한 표현** 없음 ("I don't know" 등)

In [4]:
# ---------------------------------------------------
# 규칙 기반 커스텀 평가자 구현
# ---------------------------------------------------
# ============================================================
# TODO: (Run, Example) -> dict 형식의 규칙 기반 평가자 함수를 완성하세요
# 힌트: run.outputs.get("answer", ""), 길이/키워드/불필요표현 3가지 기준으로 점수 합산
# 예상 결과: {"key": "heuristic_score", "score": 0~1}
# ============================================================

from langsmith.schemas import Run, Example

def length_and_keyword_evaluator(run: Run, example: Example) -> dict:
    """
    답변 길이와 키워드 포함 여부를 평가하는 규칙 기반 평가자
    
    평가 기준:
    - 길이: 50-500자 사이가 적절 (최대 0.4점)
    - 핵심 키워드: Transformer 관련 용어 포함 (최대 0.4점)
    - 불필요한 표현: "I don't know", "unclear" 등 없음 (최대 0.2점)
    
    총점: 1.0점
    """
    answer = run.outputs.get("answer", "")
    
    score = 0.0
    
    # 1단계: 길이 평가 (0.0 ~ 0.4)
    length = len(answer)
    if 50 <= length <= 500:
        score += 0.4
    elif 30 <= length < 50 or 500 < length <= 700:
        score += 0.2
    elif length < 30:
        score += 0.0
    else:
        score += 0.1
    
    # 2단계: 키워드 평가 (0.0 ~ 0.4) — Transformer 관련 핵심 용어 포함 여부
    important_keywords = [
        "트랜스포머", "어텐션", "인코더", "디코더",
        "셀프 어텐션", "멀티헤드", "레이어", "메커니즘"
    ]
    answer_lower = answer.lower()
    keyword_count = sum(1 for kw in important_keywords if kw in answer_lower)
    
    if keyword_count >= 3:
        score += 0.4
    elif keyword_count >= 2:
        score += 0.3
    elif keyword_count >= 1:
        score += 0.2
    
    # 3단계: 불필요한 표현 체크 (0.0 ~ 0.2)
    negative_phrases = [
        "잘 모르겠", "불분명", "확실하지 않",
        "답변할 수 없", "정보가 없", "알 수 없"
    ]
    has_negative = any(phrase in answer_lower for phrase in negative_phrases)
    
    if not has_negative:
        score += 0.2
    
    return {"key": "heuristic_score", "score": score}

> ⚠️ **자주 하는 실수**: LLM 기반 평가자에서 예외 처리를 빠뜨리면 API 오류 하나가 전체 평가 실행을 중단시킵니다. `try/except` 블록으로 감싸고 오류 시 `score: 0.0`을 반환해 평가가 계속 진행되도록 하세요. `score`는 반드시 0~1 사이 float이어야 합니다.

### 규칙 기반 평가자 테스트

실제 LangSmith 실험 전에 Mock 객체로 동작을 미리 확인해볼게요.

> 🔑 **핵심 개념**: 임베딩 기반 평가는 두 텍스트를 **한 번의 API 호출**로 임베딩해 코사인 유사도를 계산합니다. LLM 호출보다 10배 이상 저렴하고 빠릅니다. 단, 참조 답변이 반드시 있어야 하고, 표현은 달라도 의미가 같은 경우에 특히 유용합니다.

In [5]:
# ---------------------------------------------------
# Mock 객체로 규칙 기반 평가자 동작 검증
# ---------------------------------------------------
# 테스트용 Mock 객체 생성
class MockRun:
    def __init__(self, outputs):
        self.outputs = outputs

class MockExample:
    def __init__(self, inputs, outputs):
        self.inputs = inputs
        self.outputs = outputs

print("=" * 60)
print("규칙 기반 평가자 테스트")
print("=" * 60)

# 테스트 케이스 1: 좋은 답변
print("\n[테스트 1] 좋은 답변")
good_run = MockRun({
    "answer": "트랜스포머 아키텍처는 인코더와 디코더 모두에서 셀프 어텐션 메커니즘을 사용합니다. 멀티헤드 어텐션을 통해 모델이 여러 위치에 동시에 집중할 수 있어 시퀀스 처리에 매우 효과적입니다."
})
example = MockExample({"question": "트랜스포머란 무엇인가요?"}, {})

result = length_and_keyword_evaluator(good_run, example)
print(f"답변: {good_run.outputs['answer']}")
print(f"점수: {result['score']:.2f}")

# 테스트 케이스 2: 나쁜 답변
print("\n[테스트 2] 나쁜 답변 (너무 짧음)")
bad_run = MockRun({"answer": "잘 모르겠습니다."})
result = length_and_keyword_evaluator(bad_run, example)
print(f"답변: {bad_run.outputs['answer']}")
print(f"점수: {result['score']:.2f}")

# 테스트 케이스 3: 중간 품질 답변
print("\n[테스트 3] 중간 품질 답변 (키워드 부족)")
medium_run = MockRun({
    "answer": "자연어 처리 작업에 사용되는 신경망 모델입니다."
})
result = length_and_keyword_evaluator(medium_run, example)
print(f"답변: {medium_run.outputs['answer']}")
print(f"점수: {result['score']:.2f}")

규칙 기반 평가자 테스트

[테스트 1] 좋은 답변
답변: The Transformer architecture uses self-attention mechanisms in both encoder and decoder layers. Multi-head attention allows the model to focus on different positions simultaneously, making it highly effective for sequence processing.
점수: 1.00

[테스트 2] 나쁜 답변 (너무 짧음)
답변: I don't know.
점수: 0.00

[테스트 3] 중간 품질 답변 (키워드 부족)
답변: It is a neural network model used in natural language processing tasks.
점수: 0.60


---

## 2. LLM 기반 커스텀 평가자

LLM을 직접 평가자로 활용해요. 규칙으로 포착하기 어려운 **근거성(Groundedness)** — 답변이 컨텍스트에만 근거하는지 — 을 판단합니다.

> **주의**: LLM 기반 평가자는 매 평가마다 LLM 호출이 발생해서 추가 비용이 들어요. 개발 초기에는 규칙 기반으로 먼저 테스트하고, 정밀 평가 시 LLM 기반을 사용하는 것이 좋아요.

In [6]:
# ---------------------------------------------------
# LLM 기반 커스텀 평가자 구현 (Groundedness)
# ---------------------------------------------------
# ============================================================
# TODO: LLM 평가 체인을 만들고 0~10 점수를 0~1로 정규화하는 평가자를 완성하세요
# 힌트: evaluation_chain.invoke({...})로 점수 문자열 받고, re.findall(r'\d+', ...)로 숫자 추출
# 예상 결과: {"key": "llm_groundedness", "score": 0~1}
# ============================================================

from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
import openai

# 평가용 LLM 체인 생성
evaluation_prompt = PromptTemplate.from_template(
    """당신은 전문 평가자입니다. 답변이 주어진 컨텍스트에 근거하는지 평가하세요.

컨텍스트:
{context}

질문: {question}

답변:
{answer}

평가 기준:
- 0-3: 답변이 컨텍스트와 모순되거나 근거가 없음 (할루시네이션)
- 4-6: 부분적으로 근거가 있으나 일부 정보가 부정확하거나 불완전
- 7-9: 대부분 정확하고 컨텍스트에 근거함
- 10: 모든 정보가 컨텍스트에서 직접 확인됨

숫자 점수(0-10)만 답하세요. 설명은 불필요합니다:"""
)

# temperature=0: 일관된 점수 반환
evaluator_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
evaluation_chain = evaluation_prompt | evaluator_llm | StrOutputParser()

def llm_groundedness_evaluator(run: Run, example: Example) -> dict:
    """
    LLM을 사용하여 답변이 컨텍스트에 근거하는지 평가
    """
    answer = run.outputs.get("answer", "")
    context = run.outputs.get("context", "")
    question = run.outputs.get("question", "")
    
    if not context or not answer:
        return {"key": "llm_groundedness", "score": 0.0}
    
    try:
        score_str = evaluation_chain.invoke({
            "context": context,
            "question": question,
            "answer": answer
        })
        
        import re
        numbers = re.findall(r'\b(\d{1,2})\b', score_str)
        if numbers:
            score = int(numbers[0])
            # 0~10 점수를 0~1로 정규화
            normalized_score = min(max(score / 10.0, 0.0), 1.0)
        else:
            normalized_score = 0.0
    except (openai.APIError, openai.APIConnectionError, openai.RateLimitError) as e:
        print(f"평가 오류 (API): {e}")
        normalized_score = 0.0
    except ValueError as e:
        print(f"평가 오류 (파싱): {e}")
        normalized_score = 0.0
    
    return {"key": "llm_groundedness", "score": normalized_score}

---

## 3. 임베딩 거리 기반 평가자

생성된 답변과 참조 답변의 **코사인 유사도(Cosine Similarity)**를 측정해요. LLM 호출보다 저렴하고, 의미적 유사도를 측정할 수 있어서 유용합니다.

참조 답변이 있는 경우에만 사용할 수 있어요.

In [7]:
# ---------------------------------------------------
# 임베딩 거리 기반 커스텀 평가자 구현
# ---------------------------------------------------
# ============================================================
# TODO: 코사인 유사도를 계산하는 임베딩 평가자를 완성하세요
# 힌트: embedding_model.embed_documents([generated, reference])로 두 벡터를 얻고, np.dot() / (norm * norm)으로 계산
# 예상 결과: {"key": "embedding_similarity", "score": 0~1}
# ============================================================

import openai  # except 절에서 openai.APIError 등을 참조하므로 명시적 import
from langchain_openai import OpenAIEmbeddings
import numpy as np

# 임베딩 모델 초기화
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

def cosine_similarity(vec1, vec2):
    """두 벡터의 코사인 유사도 계산"""
    dot_product = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    return dot_product / (norm1 * norm2)

def embedding_similarity_evaluator(run: Run, example: Example) -> dict:
    """
    임베딩 코사인 유사도로 답변과 참조 답변 비교
    """
    generated_answer = run.outputs.get("answer", "")
    reference_answer = example.outputs.get("answer", "")
    
    if not reference_answer or not generated_answer:
        return {"key": "embedding_similarity", "score": 0.0}
    
    try:
        # 두 텍스트를 한 번에 임베딩 (API 호출 최소화)
        embeddings = embedding_model.embed_documents([
            generated_answer,
            reference_answer
        ])
        
        similarity = cosine_similarity(
            np.array(embeddings[0]),
            np.array(embeddings[1])
        )
        
        # 텍스트 임베딩의 코사인 유사도는 거의 항상 [0, 1] 범위입니다.
        # (음의 값이 나오는 경우는 극히 드뭅니다.)
        # 따라서 별도의 [-1,1] → [0,1] 정규화 없이 클리핑만 적용합니다.
        normalized_score = float(np.clip(similarity, 0.0, 1.0))
        
    except (openai.APIError, openai.APIConnectionError, openai.RateLimitError) as e:
        print(f"임베딩 평가 오류 (API): {e}")
        normalized_score = 0.0
    except (ValueError, np.linalg.LinAlgError) as e:
        print(f"임베딩 평가 오류 (계산): {e}")
        normalized_score = 0.0
    
    return {"key": "embedding_similarity", "score": normalized_score}

> 💡 **실무 팁**: 코사인 유사도의 이론적 범위는 [-1, 1]이지만, 텍스트 임베딩(OpenAI, Sentence-Transformers 등)에서는 거의 항상 **[0, 1]** 범위의 값이 나옵니다. `(similarity + 1) / 2`로 정규화하면 0.85 같은 점수가 0.925로 부풀려져 실제 품질과 괴리가 생깁니다. 텍스트 임베딩을 사용할 때는 `np.clip(similarity, 0, 1)`로 클리핑만 하는 것이 더 직관적입니다.

---

## LangSmith 데이터셋 준비

평가 실행 전에 LangSmith에 데이터셋을 올려요. 이번에는 참조 답변도 포함합니다 (임베딩 기반 평가자가 사용).

In [8]:
from langsmith import Client
from langsmith import utils as ls_utils

client = Client()
dataset_name = "transformer-qa-custom-eval"

# 평가용 데이터 (질문 + 참조 답변)
qa_pairs = [
    {
        "question": "트랜스포머 아키텍처란 무엇인가요?",
        "answer": "트랜스포머는 순환이나 합성곱 없이 어텐션 메커니즘에만 전적으로 의존하는 신경망 아키텍처입니다."
    },
    {
        "question": "셀프 어텐션은 어떻게 작동하나요?",
        "answer": "셀프 어텐션은 시퀀스 내 모든 위치 간의 어텐션 가중치를 계산하여, 모델이 서로 다른 표현 부분공간의 정보를 동시에 참조할 수 있게 합니다."
    },
    {
        "question": "트랜스포머가 RNN보다 유리한 점은 무엇인가요?",
        "answer": "트랜스포머는 시퀀스를 병렬로 처리할 수 있고, 장거리 의존성을 더 효과적으로 포착하며, RNN보다 훈련 효율이 높습니다."
    },
    {
        "question": "멀티헤드 어텐션이란 무엇인가요?",
        "answer": "멀티헤드 어텐션은 여러 개의 어텐션 메커니즘을 병렬로 적용하여, 모델이 서로 다른 표현 부분공간의 정보를 동시에 참조할 수 있게 합니다."
    },
    {
        "question": "포지셔널 인코딩의 역할은 무엇인가요?",
        "answer": "포지셔널 인코딩은 트랜스포머에 단어의 위치 정보를 제공하여, 순환 구조 없이도 시퀀스의 순서를 파악할 수 있게 합니다."
    },
    {
        "question": "스케일드 닷 프로덕트 어텐션이란 무엇인가요?",
        "answer": "스케일드 닷 프로덕트 어텐션은 쿼리와 키의 내적을 키 차원의 제곱근으로 나눈 뒤 소프트맥스를 적용하여 어텐션 가중치를 계산하는 방식입니다."
    },
]

# 데이터셋 생성 또는 사용
try:
    dataset = client.read_dataset(dataset_name=dataset_name)
    print(f"✅ 기존 데이터셋 사용: {dataset_name}")
except ls_utils.LangSmithNotFoundError:
    dataset = client.create_dataset(
        dataset_name=dataset_name,
        description="트랜스포머 Q&A 커스텀 평가자 테스트용"
    )
    
    for qa in qa_pairs:
        client.create_example(
            inputs={"question": qa["question"]},
            outputs={"answer": qa["answer"]},
            dataset_id=dataset.id
        )
    print(f"✅ 새 데이터셋 생성: {dataset_name} ({len(qa_pairs)}개 예제)")


✅ 새 데이터셋 생성: transformer-qa-custom-eval (4개 예제)


---

## 평가 실행

세 가지 커스텀 평가자를 조합해서 종합 평가를 실행합니다.

In [9]:
# ---------------------------------------------------
# 3가지 커스텀 평가자 조합하여 종합 평가 실행
# ---------------------------------------------------
# ============================================================
# TODO: evaluate()에 세 가지 평가자 리스트를 전달하여 평가를 실행하세요
# 힌트: evaluators=[length_and_keyword_evaluator, llm_groundedness_evaluator, embedding_similarity_evaluator]
# 예상 결과: experiment_results.experiment_name 출력
# ============================================================

from langsmith.evaluation import evaluate

# 평가 실행
experiment_results = evaluate(
    rag_answer_with_context,
    data=dataset_name,
    evaluators=[
        length_and_keyword_evaluator,      # 규칙 기반: 빠르고 비용 없음
        llm_groundedness_evaluator,        # LLM 기반: 근거성 판단
        embedding_similarity_evaluator     # 임베딩 기반: 의미 유사도
    ],
    experiment_prefix="custom-evaluators",
    metadata={
        "model": "gpt-4o-mini",
        "embedding": "text-embedding-3-small",
        "evaluator_types": "heuristic, llm, embedding"
    }
)

print("\n" + "=" * 60)
print("평가 완료!")
print("=" * 60)
print(f"\n결과 실험 이름: {experiment_results.experiment_name}")
print("\nLangSmith 대시보드에서 다음 메트릭을 확인할 수 있습니다:")
print("  - heuristic_score: 규칙 기반 점수")
print("  - llm_groundedness: LLM이 판단한 근거성")
print("  - embedding_similarity: 참조 답변과의 유사도")

View the evaluation results for experiment: 'custom-evaluators-58db0aa0' at:
https://smith.langchain.com/o/7aba17aa-11f3-44e7-8a89-ca2e8b897dcb/datasets/fd0ba04e-0a6e-41c1-ab31-7385fa3f948b/compare?selectedSessions=c67ada05-526a-467e-bde9-488b3b7335fe




0it [00:00, ?it/s]


평가 완료!

결과 실험 이름: custom-evaluators-58db0aa0

LangSmith 대시보드에서 다음 메트릭을 확인할 수 있습니다:
  - heuristic_score: 규칙 기반 점수
  - llm_groundedness: LLM이 판단한 근거성
  - embedding_similarity: 참조 답변과의 유사도


---

## 평가 결과 해석

세 가지 커스텀 평가자의 결과를 질문별로 확인해볼게요. 각 평가자가 같은 답변을 어떻게 다르게 평가하는지 비교하는 것이 핵심입니다.

In [ ]:
# ---------------------------------------------------
# 질문별 평가 결과 상세 분석
# ---------------------------------------------------
import pandas as pd

rows = []
for result in experiment_results:
    question = result["example"].inputs["question"]
    
    scores = {}
    for eval_result in result["evaluation_results"]["results"]:
        scores[eval_result.key] = eval_result.score
    
    rows.append({
        "질문": question[:20] + "...",
        "heuristic": f"{scores.get('heuristic_score', 0):.2f}",
        "llm_groundedness": f"{scores.get('llm_groundedness', 0):.2f}",
        "embedding_sim": f"{scores.get('embedding_similarity', 0):.2f}",
    })

df_result = pd.DataFrame(rows)
print("=" * 70)
print("커스텀 평가자 결과 비교 (질문별)")
print("=" * 70)
print(df_result.to_string(index=False))

# 평균 점수
print("\n" + "-" * 70)
print("평균 점수:")
for col in ["heuristic", "llm_groundedness", "embedding_sim"]:
    avg = sum(float(r[col]) for r in rows) / len(rows)
    print(f"  {col:20s}: {avg:.2f}")


### 실제 실행 결과 해석

위 코드를 실행하면 다음과 유사한 결과를 얻을 수 있습니다:

| 질문 | heuristic | llm_groundedness | embedding_sim |
|------|:---------:|:----------------:|:-------------:|
| 트랜스포머 아키텍처란? | 1.00 | 1.00 | 0.54 |
| 셀프 어텐션은 어떻게? | 1.00 | 1.00 | 0.76 |
| 멀티헤드 어텐션이란? | 1.00 | 1.00 | 0.81 |
| 포지셔널 인코딩의 역할은? | 0.80 | 1.00 | 0.78 |

### 세 가지 평가자의 특성 차이

| 평가자 | 측정 대상 | 점수 해석 |
|--------|----------|----------|
| **heuristic_score** | 답변 길이, 핵심 키워드 포함, 불필요 표현 유무 | 1.0에 가까울수록 형식적으로 좋은 답변. 내용의 정확성은 판단 불가 |
| **llm_groundedness** | 답변이 검색된 컨텍스트에 근거하는지 | 0~1 연속 점수. LLM이 "이 답변이 컨텍스트에서 나온 정보인가?"를 판단 |
| **embedding_similarity** | 생성 답변과 참조 답변의 의미적 유사도 | 1.0에 가까울수록 참조 답변과 의미가 유사. 0.8 이상이면 매우 유사 |

**결과에서 주목할 점:**

- `heuristic_score`는 대부분 1.0 — 키워드가 잘 포함되고 적절한 길이의 답변이기 때문. 하지만 이것만으로는 답변이 맞는지 알 수 없어요.
- `llm_groundedness`가 전부 1.0 — RAG 시스템이 컨텍스트에 충실하게 답변하고 있다는 뜻. 할루시네이션이 없습니다.
- `embedding_similarity`가 0.54~0.81로 편차가 큼 — RAG 답변이 참조 답변보다 더 상세하거나 표현이 다르면 유사도가 낮아질 수 있어요. 특히 "트랜스포머 아키텍처" 질문(0.54)은 RAG가 훨씬 상세한 답변을 생성해서 참조 답변과 벡터 거리가 벌어진 것입니다.

> 💡 **실무 팁**: `heuristic_score`가 높지만 `llm_groundedness`가 낮으면 → 형식은 갖추었지만 할루시네이션이 있는 답변입니다. `llm_groundedness`가 높지만 `embedding_similarity`가 낮으면 → 컨텍스트에 근거하지만 참조 답변과 다른 관점에서 답변한 것입니다.

---

## 정리

### 커스텀 평가자 3가지 유형 비교

| 유형 | 장점 | 단점 | 적합한 사용 사례 |
|------|------|------|----------------|
| **규칙 기반** | 빠름, 비용 없음, 투명함 | 단순한 판단만 가능 | 길이, 키워드, 형식 체크 |
| **LLM 기반** | 정교한 판단, 맥락 이해 | 비용 발생, 일관성 낮음 | 근거성, 품질, 논리성 평가 |
| **임베딩 기반** | 의미 유사도, LLM보다 저렴 | 참조 답변 필수 | 정답과의 유사도 측정 |

### 평가자 조합 전략

- **개발 초기 (빠른 반복)**: 규칙 기반만
- **중간 점검 (정밀도 필요)**: 규칙 기반 + 임베딩 기반
- **최종 평가 (모든 측면)**: 세 가지 모두

### 커스텀 평가자 작성 체크리스트

1. `(Run, Example) -> dict` 시그니처 준수
2. `score`는 반드시 0~1 사이 값
3. `run.outputs`와 `example.outputs` 존재 여부 검증
4. 예외 발생 시 0 반환 (평가 중단 방지)
5. `key` 이름을 직관적으로 (`heuristic_score`, `llm_groundedness` 등)

### 다음 노트북 예고

다음에는 LLM 호출 없이 통계적 방법으로 평가하는 **Heuristic 메트릭** — ROUGE, BLEU, METEOR, SemScore — 을 학습해요.